# Лекция 7. Разработка веб-приложений и чат-ботов для доступа к юридической информации

### Цель лекции

Изучить процесс разработки веб-приложений и чат-ботов для предоставления доступа к юридической информации, обсудить архитектуру системы, выбор технологического стека, а также рассмотреть практические примеры реализации на языке Python.

---

### Введение

Информационно-поисковые системы, рассмотренные в предшествующих лекциях курса, представляют собой сложные программные комплексы, включающие базы данных, NLP-конвейеры, классификаторы и поисковые модули. Однако практическая ценность этих компонентов определяется не столько их внутренним устройством, сколько тем, насколько удобно и эффективно конечный пользователь может взаимодействовать с системой. Именно пользовательский интерфейс выступает связующим звеном между аналитическим ядром системы и человеком, формулирующим информационную потребность.

В современной практике доминируют два основных канала взаимодействия с юридическими информационными системами. Первый канал представлен веб-приложениями, предоставляющими графический интерфейс в браузере с функциями поиска, фильтрации, просмотра документов и аналитических дашбордов. Такие системы как «КонсультантПлюс», «Гарант», «Кад.Арбитр» являются примерами промышленных веб-приложений, обеспечивающих доступ к миллионам юридических документов. Второй канал представлен диалоговыми интерфейсами (чат-ботами), реализующими взаимодействие на естественном языке через мессенджеры (Telegram, WhatsApp) или встроенные виджеты на веб-сайтах.

Оба канала основаны на архитектуре «клиент-сервер», в которой серверная часть (backend) инкапсулирует бизнес-логику, обеспечивает доступ к базе данных и поисковым индексам, а клиентская часть (frontend) отвечает за представление данных и взаимодействие с пользователем. Коммуникация между клиентом и сервером осуществляется посредством программного интерфейса (API), реализующего стандартизованный протокол обмена данными.

Разработка пользовательского интерфейса для юридической информационной системы ставит ряд специфических задач. С точки зрения информационной архитектуры необходимо обеспечить эффективную навигацию по большим объёмам документов с многоуровневой системой фильтрации. С точки зрения юзабилити интерфейс должен быть доступен как профессиональным юристам, привыкшим к сложным поисковым запросам, так и гражданам без специальной подготовки. С точки зрения безопасности система должна обеспечивать защиту персональных данных участников судебных процессов в соответствии с требованиями Федерального закона № 152-ФЗ.

Настоящая лекция последовательно рассматривает архитектуру клиент-серверных юридических систем, технологический стек разработки, реализацию серверной и клиентской частей, создание диалоговых интерфейсов, обеспечение безопасности, тестирование и развёртывание. Каждый раздел сопровождается демонстрационным кодом, иллюстрирующим рассмотренные концепции.

---


## 1. Основные понятия и определения

Прежде чем перейти к практическим аспектам разработки, необходимо установить единый терминологический аппарат, охватывающий ключевые понятия веб-разработки и проектирования диалоговых систем.

### 1.1. Веб-приложение

Веб-приложение представляет собой программную систему, пользовательский интерфейс которой реализован в среде веб-браузера и доступен через протокол HTTP/HTTPS. В отличие от классических настольных приложений, веб-приложения не требуют установки на клиентском устройстве и обеспечивают доступ к функциональности из любого места при наличии сетевого подключения. Данное свойство особенно ценно для юридических систем, пользователи которых могут находиться в судах, офисах, командировках.

Современные веб-приложения подразделяются на многостраничные (Multi-Page Application, MPA), в которых каждая навигация приводит к загрузке новой HTML-страницы с сервера, и одностраничные (Single-Page Application, SPA), в которых весь интерфейс загружается однократно, а дальнейшее взаимодействие реализуется через динамическое обновление содержимого без перезагрузки страницы. SPA-подход, реализуемый с помощью фреймворков React, Vue.js или Angular, обеспечивает более отзывчивый интерфейс, что критически важно для систем интерактивного поиска, где пользователь итеративно уточняет запрос на основе промежуточных результатов.

### 1.2. Чат-бот

Чат-бот (диалоговый агент) — программная система, реализующая взаимодействие с пользователем посредством обмена текстовыми или голосовыми сообщениями на естественном языке. В контексте юридической информационной системы чат-бот выступает альтернативным каналом доступа к аналитическим функциям, ориентированным на пользователей, предпочитающих диалоговый формат взаимодействия.

Архитектурно чат-боты подразделяются на несколько категорий. Правиловые (rule-based) боты реагируют на ключевые слова или команды по заранее определённым сценариям; их возможности ограничены, но поведение полностью предсказуемо. Боты с поддержкой NLP используют модели обработки естественного языка для распознавания намерения пользователя (intent detection) и извлечения параметров запроса (slot filling), что позволяет обрабатывать запросы, сформулированные в произвольной форме. Генеративные боты, основанные на больших языковых моделях (LLM), способны формировать развёрнутые ответы, однако требуют тщательного контроля достоверности генерируемой информации, что особенно критично в юридическом контексте, где недостоверная информация может повлечь серьёзные последствия.

### 1.3. API (Application Programming Interface)

Программный интерфейс приложения (API) определяет набор операций, форматов данных и протоколов, посредством которых программные компоненты взаимодействуют друг с другом. В контексте веб-разработки наибольшее распространение получили REST API (Representational State Transfer), основанные на протоколе HTTP и использующие стандартные методы ($\texttt{GET}$, $\texttt{POST}$, $\texttt{PUT}$, $\texttt{DELETE}$) для выполнения операций CRUD (Create, Read, Update, Delete).

Формально REST API может быть описан как функция $f: \mathcal{R} \to \mathcal{S}$, отображающая множество допустимых HTTP-запросов $\mathcal{R}$ (определяемых методом, URL-путём, заголовками и телом запроса) в множество ответов $\mathcal{S}$ (определяемых кодом состояния, заголовками и телом ответа). Каждый эндпойнт API представляет собой конкретную точку входа, идентифицируемую парой (метод, URL-путь), например $\texttt{GET /api/v1/documents}$ или $\texttt{POST /api/v1/search}$.

Альтернативным подходом является GraphQL, позволяющий клиенту декларативно описывать структуру запрашиваемых данных, что устраняет проблему избыточной или недостаточной выборки (over-fetching / under-fetching), характерную для REST API. Однако для юридических систем с относительно стабильной структурой запросов REST API остаётся более распространённым и практичным выбором.

### 1.4. Архитектура клиент-сервер

Архитектура клиент-сервер представляет собой модель организации распределённых вычислений, в которой функции разделены между двумя типами участников: клиентом, инициирующим запросы и отображающим результаты, и сервером, обрабатывающим запросы и предоставляющим ресурсы. Данная модель обеспечивает централизованное управление данными (все документы хранятся на сервере), масштабируемость (один сервер обслуживает множество клиентов), а также разделение ответственности между компонентами.

В юридических информационных системах типичная архитектура включает три уровня (three-tier architecture). Уровень представления (presentation tier) реализует пользовательский интерфейс в виде веб-приложения или чат-бота. Уровень бизнес-логики (application tier) содержит серверное приложение, реализующее обработку запросов, валидацию данных, вызов NLP-моделей и формирование ответов. Уровень данных (data tier) представлен СУБД, полнотекстовыми и векторными индексами, рассмотренными в Лекции 6.

---


## 2. Архитектура системы

### 2.1. Общая схема юридической информационной системы

Архитектура юридической информационно-поисковой системы определяется совокупностью компонентов, их функциональной специализацией и протоколами взаимодействия. Грамотное архитектурное решение обеспечивает масштабируемость (возможность наращивания производительности при росте нагрузки), отказоустойчивость (продолжение функционирования при выходе из строя отдельных компонентов), сопровождаемость (возможность модификации компонентов без влияния на остальные части системы) и безопасность (защиту данных и контроль доступа).

Общая схема системы включает несколько функциональных слоёв. Слой представления объединяет все клиентские интерфейсы: веб-приложение (SPA), мобильное приложение, чат-бот в мессенджере, а также внешние системы, обращающиеся к API программно. Слой API-шлюза (API Gateway) обеспечивает единую точку входа для всех клиентов, реализуя маршрутизацию запросов, аутентификацию, ограничение частоты запросов (rate limiting) и журналирование. Слой бизнес-логики содержит микросервисы или модули монолитного приложения, реализующие конкретные функции: поиск документов, NER-обработку, классификацию, суммаризацию. Слой данных включает реляционную СУБД (PostgreSQL), полнотекстовый индекс, векторное хранилище и кэш (Redis).

Взаимодействие между слоями осуществляется по принципу нисходящей зависимости: каждый слой обращается только к нижележащему и не имеет прямого доступа к вышележащему. Данный принцип обеспечивает слабую связанность компонентов и возможность их независимой замены или масштабирования. Горизонтальная коммуникация между компонентами одного слоя осуществляется через брокер сообщений (RabbitMQ, Kafka) для асинхронных задач или через прямые HTTP-вызовы для синхронных операций.

### 2.2. Паттерны проектирования

При проектировании серверной части юридической информационной системы применяются несколько архитектурных паттернов. Паттерн «репозиторий» (Repository) инкапсулирует логику доступа к данным, предоставляя бизнес-логике абстрактный интерфейс для работы с сущностями предметной области без привязки к конкретной СУБД. Паттерн «фасад» (Facade) обеспечивает упрощённый интерфейс к подсистеме NLP-обработки, скрывая сложность взаимодействия между NER-моделью, классификатором и суммаризатором. Паттерн «стратегия» (Strategy) позволяет динамически переключаться между различными алгоритмами поиска: полнотекстовым (BM25), семантическим (на основе эмбеддингов) и гибридным.

В контексте REST API существенную роль играет паттерн DTO (Data Transfer Object), определяющий структуру данных, передаваемых между клиентом и сервером. В экосистеме Python/FastAPI DTO реализуются посредством Pydantic-моделей, обеспечивающих автоматическую валидацию и сериализацию данных. Строгая типизация DTO снижает вероятность ошибок интеграции между frontend- и backend-компонентами.

---


In [ ]:
# ============================================================
# Визуализация трёхуровневой архитектуры юридической системы
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 9)
ax.axis('off')
ax.set_title('Архитектура юридической информационно-поисковой системы',
             fontsize=14, fontweight='bold')

# --- Слой представления (клиенты) ---
clients = [
    (0.5, 7.5, 3.0, 1.2, 'Веб-приложение\n(React SPA)', '#E3F2FD'),
    (4.0, 7.5, 3.0, 1.2, 'Telegram-бот\n(aiogram)', '#FFF9C4'),
    (7.5, 7.5, 3.0, 1.2, 'Мобильное\nприложение', '#F3E5F5'),
    (11.0, 7.5, 2.5, 1.2, 'Внешние\nсистемы', '#ECEFF1'),
]
for x, y, w, h, label, color in clients:
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                    edgecolor='#333', facecolor=color, lw=1.5)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2, label, ha='center', va='center', fontsize=9)

ax.text(7, 7.1, 'Слой представления (Presentation Tier)',
        ha='center', fontsize=11, style='italic', color='#1565C0')

# --- API Gateway ---
gw = mpatches.FancyBboxPatch((2, 5.8), 10, 0.9, boxstyle="round,pad=0.08",
                              edgecolor='#333', facecolor='#BBDEFB', lw=2)
ax.add_patch(gw)
ax.text(7, 6.25, 'API Gateway  (аутентификация, маршрутизация, rate limiting)',
        ha='center', va='center', fontsize=10, fontweight='bold')

# Стрелки клиенты -> gateway
for x_c in [2.0, 5.5, 9.0, 12.25]:
    ax.annotate('', xy=(min(x_c, 12), 6.7), xytext=(x_c, 7.5),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#1565C0'))

# --- Слой бизнес-логики ---
services = [
    (0.5, 3.8, 3.0, 1.5, 'Сервис поиска\n(FTS + семантический\n+ гибридный)', '#C8E6C9'),
    (4.0, 3.8, 3.0, 1.5, 'Сервис NLP\n(NER, классификация,\nсуммаризация)', '#FFF9C4'),
    (7.5, 3.8, 3.0, 1.5, 'Сервис документов\n(CRUD, пагинация,\nфильтрация)', '#FFCCBC'),
    (11.0, 3.8, 2.5, 1.5, 'Сервис\nуведомлений\n(WebSocket)', '#D1C4E9'),
]
for x, y, w, h, label, color in services:
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                    edgecolor='#333', facecolor=color, lw=1.5)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2, label, ha='center', va='center', fontsize=8)

ax.text(7, 3.4, 'Слой бизнес-логики (Application Tier)',
        ha='center', fontsize=11, style='italic', color='#2E7D32')

# Стрелки gateway -> services
for x_s in [2.0, 5.5, 9.0, 12.25]:
    ax.annotate('', xy=(min(x_s, 12), 5.3), xytext=(x_s, 5.8),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#2E7D32'))

# --- Слой данных ---
data_stores = [
    (0.5, 1.0, 2.8, 1.8, 'PostgreSQL\n(метаданные,\nFTS-индекс)', '#E8EAF6'),
    (3.8, 1.0, 2.8, 1.8, 'Векторное\nхранилище\n(pgvector / FAISS)', '#E0F7FA'),
    (7.1, 1.0, 2.8, 1.8, 'Redis\n(кэш запросов,\nсессии)', '#FBE9E7'),
    (10.4, 1.0, 3.0, 1.8, 'Object Storage\n(S3 / MinIO:\nполные тексты)', '#F1F8E9'),
]
for x, y, w, h, label, color in data_stores:
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
                                    edgecolor='#333', facecolor=color, lw=1.5)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2, label, ha='center', va='center', fontsize=8)

ax.text(7, 0.6, 'Слой данных (Data Tier)',
        ha='center', fontsize=11, style='italic', color='#BF360C')

# Стрелки services -> data
for x_d in [1.9, 5.2, 8.5, 11.9]:
    ax.annotate('', xy=(x_d, 2.8), xytext=(x_d, 3.8),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='#BF360C'))

plt.tight_layout()
plt.show()
print("Трёхуровневая архитектура юридической ИПС")

## 3. Выбор технологического стека

Технологический стек — совокупность программных инструментов, фреймворков, библиотек и платформ, используемых при разработке информационной системы. Выбор стека определяется требованиями к производительности, экспертизой команды разработчиков, зрелостью экосистемы и долгосрочной поддержкой. В контексте юридической информационно-поисковой системы ключевым фактором является совместимость с экосистемой машинного обучения и обработки естественного языка, поскольку серверная часть тесно интегрирована с NLP-конвейером.

### 3.1. Backend-технологии

Язык Python является доминирующим выбором для серверной части юридических аналитических систем благодаря нескольким обстоятельствам. Во-первых, Python обладает наиболее развитой экосистемой для машинного обучения и NLP (scikit-learn, PyTorch, Hugging Face Transformers, spaCy, DeepPavlov), что обеспечивает бесшовную интеграцию серверного приложения с аналитическими компонентами. Во-вторых, современные асинхронные фреймворки (FastAPI, Starlette) обеспечивают производительность, достаточную для большинства юридических приложений. В-третьих, SQLAlchemy и Alembic предоставляют зрелый инструментарий для работы с реляционными базами данных.

Среди Python-фреймворков для создания REST API FastAPI представляет собой наиболее обоснованный выбор для новых проектов. FastAPI основан на стандартах OpenAPI и JSON Schema, обеспечивает автоматическую генерацию интерактивной документации (Swagger UI), поддерживает асинхронную обработку запросов (через $\texttt{async/await}$), а также реализует строгую валидацию входных данных посредством Pydantic-моделей. Производительность FastAPI сопоставима с аналогами на Go и Node.js благодаря использованию ASGI-сервера Uvicorn.

### 3.2. Frontend-технологии

Для реализации клиентской части юридической системы наиболее распространённым подходом является создание одностраничного приложения (SPA) с использованием JavaScript-фреймворка. React (разработка Meta) предоставляет компонентную модель интерфейса с виртуальным DOM и однонаправленным потоком данных. Vue.js предлагает более низкий порог вхождения при сопоставимой функциональности. Angular (разработка Google) представляет собой полнофункциональный фреймворк, наиболее подходящий для крупных корпоративных приложений.

Взаимодействие frontend-приложения с backend API осуществляется через HTTP-запросы с использованием библиотек $\texttt{fetch}$ (встроенная в браузер) или $\texttt{axios}$ (сторонняя библиотека с расширенным функционалом). Формат обмена данными — JSON. Для обеспечения безопасности кросс-доменных запросов сервер должен быть сконфигурирован для поддержки CORS (Cross-Origin Resource Sharing), определяющего допустимые источники запросов.

### 3.3. Технологии чат-ботов

Для разработки Telegram-ботов на Python наиболее современным решением является фреймворк aiogram (версия 3.x), реализующий полностью асинхронное взаимодействие с Telegram Bot API. Aiogram поддерживает систему роутеров для организации обработчиков, FSM (Finite State Machine) для управления диалоговыми сценариями и middleware для кросс-сквозной функциональности (логирование, аутентификация). Альтернативой является библиотека python-telegram-bot, предоставляющая синхронный интерфейс, более привычный для начинающих разработчиков.

Для реализации продвинутых диалоговых сценариев с поддержкой распознавания намерений и извлечения сущностей применяются специализированные NLU-фреймворки. Rasa Open Source предоставляет полный стек для создания диалоговых систем, включая NLU-компонент для распознавания интентов и модуль управления диалогом. В юридическом контексте типичные интенты включают «поиск_документа», «статус_дела», «определение_понятия», а извлекаемые сущности — номер дела, дату, наименование суда, категорию спора.

---


## 4. Разработка серверной части (Backend)

Серверная часть юридической информационной системы представляет собой центральный компонент, инкапсулирующий бизнес-логику, обеспечивающий доступ к данным и предоставляющий программный интерфейс (API) для всех клиентских приложений. Качество проектирования и реализации серверной части непосредственно определяет производительность, надёжность и расширяемость системы в целом.

### 4.1. Создание REST API на FastAPI

FastAPI обеспечивает декларативное описание эндпойнтов API с автоматической валидацией параметров, сериализацией ответов и генерацией документации. Каждый эндпойнт определяется декоратором, указывающим HTTP-метод и URL-путь, а типы входных и выходных данных описываются Pydantic-моделями. Такой подход реализует контрактное программирование: спецификация API определяется типами и доступна в машинно-читаемом формате (OpenAPI JSON), что обеспечивает автоматическую генерацию клиентских библиотек и интерактивной документации.

Структура серверного приложения, как правило, следует паттерну разделения на слои. Слой маршрутизации (routers) содержит определения эндпойнтов и их параметров. Слой сервисов (services) реализует бизнес-логику: валидацию запросов, вызов NLP-моделей, формирование ответов. Слой доступа к данным (repositories) инкапсулирует SQL-запросы и операции с ORM. Слой схем (schemas) определяет Pydantic-модели для валидации входных данных и формирования ответов.

### 4.2. Подключение к базе данных

Интеграция FastAPI с PostgreSQL осуществляется через SQLAlchemy ORM, рассмотренную в Лекции 6. Для асинхронных приложений применяется асинхронный драйвер $\texttt{asyncpg}$ в сочетании с $\texttt{SQLAlchemy AsyncSession}$. Паттерн зависимостей (Dependency Injection) FastAPI обеспечивает управление жизненным циклом сессий базы данных: сессия создаётся при поступлении запроса и закрывается после его обработки, что гарантирует корректное освобождение ресурсов.

### 4.3. Реализация поиска

Серверная часть должна поддерживать несколько режимов поиска. Структурированный поиск по метаданным (номер дела, дата, тип документа) реализуется стандартными SQL-запросами с фильтрацией. Полнотекстовый поиск по содержанию документов использует механизм $\texttt{tsvector}$/$\texttt{tsquery}$ PostgreSQL с функцией ранжирования $\texttt{ts\_rank}$. Семантический поиск на основе эмбеддингов вычисляет косинусное сходство между вектором запроса и векторами документов, хранящимися в pgvector-индексе. Гибридный поиск комбинирует результаты полнотекстового и семантического поиска посредством алгоритма Reciprocal Rank Fusion (RRF), определяемого формулой:

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$

где $R$ -- множество ранжированных списков от различных методов поиска, $\text{rank}_r(d)$ -- позиция документа $d$ в списке $r$, а $k$ -- константа (типично $k = 60$), управляющая относительным весом высокоранжированных документов.

---


In [ ]:
# ============================================================
# Демонстрация: Pydantic-схемы и структура FastAPI-приложения
# ============================================================

from pydantic import BaseModel, Field
from datetime import date
from typing import Optional, List
from enum import Enum

class DocType(str, Enum):
    decision    = "решение"
    ruling      = "определение"
    resolution  = "постановление"
    claim       = "исковое заявление"
    appeal      = "апелляционная жалоба"

class SearchMode(str, Enum):
    fulltext  = "fulltext"
    semantic  = "semantic"
    hybrid    = "hybrid"

class DocumentResponse(BaseModel):
    id: int
    title: str
    date: date
    doc_type: str
    case_number: Optional[str] = None
    snippet: Optional[str] = None
    relevance_score: Optional[float] = None
    class Config:
        from_attributes = True

class SearchRequest(BaseModel):
    query: str = Field(..., min_length=2, max_length=500)
    mode: SearchMode = Field(default=SearchMode.hybrid)
    doc_type: Optional[DocType] = None
    date_from: Optional[date] = None
    date_to: Optional[date] = None
    limit: int = Field(default=20, ge=1, le=100)

class SearchResponse(BaseModel):
    total: int
    query: str
    mode: str
    results: List[DocumentResponse]


# --- Reciprocal Rank Fusion ---
def reciprocal_rank_fusion(lists, k=60):
    """
    Объединение нескольких ранжированных списков методом RRF.
    lists: список списков документов (каждый отсортирован по релевантности)
    k: параметр сглаживания (типично k=60)
    """
    scores = {}
    all_docs = {}
    for ranked_list in lists:
        for rank, doc_id in enumerate(ranked_list):
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank)
            all_docs[doc_id] = doc_id
    return sorted(scores, key=scores.get, reverse=True)


# Демонстрация RRF на примере
fts_results = ["doc_A", "doc_B", "doc_C", "doc_D", "doc_E"]
sem_results = ["doc_C", "doc_A", "doc_F", "doc_B", "doc_G"]

fused = reciprocal_rank_fusion([fts_results, sem_results], k=60)
print("Демонстрация Reciprocal Rank Fusion")
print("=" * 50)
print(f"FTS-результаты: {fts_results}")
print(f"Семантические:  {sem_results}")
print(f"RRF-объединение: {fused}")
print("\nДокументы, присутствующие в обоих списках (doc_A, doc_B, doc_C),")
print("получают более высокий ранг в объединённом списке.")

# Pydantic валидация
print("\nПример валидации SearchRequest:")
req = SearchRequest(query="банкротство субсидиарная ответственность", mode="hybrid", limit=10)
print(f"  query: {req.query!r}")
print(f"  mode:  {req.mode}")
print(f"  limit: {req.limit}")

## 5. Разработка клиентской части (Frontend)

Клиентская часть юридической информационной системы обеспечивает визуальное представление данных и интерактивное взаимодействие с пользователем. В современной веб-разработке доминирует подход одностраничного приложения (SPA), при котором весь код интерфейса загружается в браузер однократно, а дальнейшее обновление содержимого происходит динамически через API-запросы к серверу без перезагрузки страницы. Такой подход обеспечивает плавность навигации и мгновенный отклик на действия пользователя, что особенно важно для систем интерактивного поиска.

### 5.1. Компонентная архитектура интерфейса

Пользовательский интерфейс юридической поисковой системы декомпозируется на автономные компоненты, каждый из которых инкапсулирует свою логику отображения и управления состоянием. Типичная структура включает компонент поисковой строки (SearchBar) с поддержкой автодополнения и подсказок, компонент панели фильтров (FilterPanel) для задания временного диапазона, типа документа, юрисдикции и категории дела, компонент списка результатов (ResultsList) с пагинацией и сортировкой, компонент карточки документа (DocumentCard) с отображением метаданных, сниппета и извлечённых сущностей, а также компонент детального просмотра (DocumentViewer) с подсветкой найденных терминов и навигацией по структуре документа.

Каждый компонент управляет своим локальным состоянием, а глобальное состояние приложения (текущий запрос, результаты поиска, параметры фильтрации) управляется централизованным хранилищем. В экосистеме React для этой цели используются хуки $\texttt{useState}$ и $\texttt{useReducer}$ для локального состояния и библиотеки Redux или Zustand для глобального.

### 5.2. Взаимодействие с Backend API

Обмен данными между frontend-приложением и сервером осуществляется посредством асинхронных HTTP-запросов. Для юридической поисковой системы типичный цикл взаимодействия включает следующие этапы. Пользователь вводит поисковый запрос и задаёт параметры фильтрации. Frontend формирует POST-запрос к эндпойнту $\texttt{/api/v1/search}$ с телом в формате JSON, соответствующим схеме $\texttt{SearchRequest}$. Сервер выполняет поиск и возвращает ответ в формате $\texttt{SearchResponse}$. Frontend обновляет состояние приложения и отрисовывает список результатов.

Обработка ошибок является важной составляющей клиент-серверного взаимодействия. HTTP-коды состояния (200 -- успех, 400 -- ошибка валидации, 404 -- не найдено, 500 -- серверная ошибка) должны корректно интерпретироваться клиентским приложением и отображаться пользователю в понятной форме. Для повышения отзывчивости интерфейса применяются техники оптимистичного обновления, debounce-механизм для поисковой строки (задержка отправки запроса до прекращения ввода) и скелетонная загрузка (отображение контуров результатов до получения данных).

---


In [ ]:
# ============================================================
# Демонстрация: структура React-компонента (для изучения)
# ============================================================

react_code = """
// SearchPage.jsx -- компонент поиска юридических документов
import React, { useState, useCallback } from 'react';
import debounce from 'lodash/debounce';

export default function SearchPage() {
  const [query, setQuery] = useState('');
  const [results, setResults] = useState([]);
  const [loading, setLoading] = useState(false);

  const performSearch = useCallback(
    debounce(async (q) => {
      if (q.length < 2) return;
      setLoading(true);
      try {
        const resp = await fetch('/api/v1/search', {
          method: 'POST',
          headers: { 'Content-Type': 'application/json' },
          body: JSON.stringify({ query: q, mode: 'hybrid', limit: 20 }),
        });
        const data = await resp.json();
        setResults(data.results);
      } catch (err) { console.error(err); }
      finally { setLoading(false); }
    }, 300), []
  );

  return (
    <div>
      <h1>Поиск судебных документов</h1>
      <input value={query}
             onChange={e => { setQuery(e.target.value);
                              performSearch(e.target.value); }}
             placeholder="Введите запрос..." />
      {loading && <p>Загрузка...</p>}
      <ul>
        {results.map(d => (
          <li key={d.id}>
            <h3>{d.title}</h3>
            <span>{d.doc_type} | {d.date}</span>
          </li>
        ))}
      </ul>
    </div>
  );
}
"""

print("React-компонент страницы поиска (JSX)")
print("=" * 60)
print(react_code)
print("Паттерны: debounce 300мс, skeleton loader, обработка ошибок HTTP")

## 6. Разработка чат-бота

Диалоговый интерфейс (чат-бот) представляет собой альтернативный канал доступа к юридической информации, ориентированный на пользователей, предпочитающих формулировать запросы на естественном языке в привычной среде мессенджера. В отличие от веб-интерфейса, чат-бот не требует от пользователя знакомства с системой фильтров и параметров поиска: информационная потребность выражается в свободной текстовой форме, а бот самостоятельно интерпретирует запрос и формирует структурированный ответ.

### 6.1. Архитектура диалогового агента

Архитектура юридического чат-бота включает несколько функциональных компонентов. Транспортный уровень обеспечивает взаимодействие с API мессенджера (Telegram Bot API), обрабатывая входящие сообщения и отправляя ответы. Уровень NLU (Natural Language Understanding) выполняет распознавание намерения пользователя (intent classification) и извлечение параметров запроса (entity extraction). Уровень управления диалогом определяет текущее состояние беседы и выбирает следующее действие. Уровень интеграции взаимодействует с backend API юридической системы, передавая структурированный запрос и получая результаты.

Формально процесс обработки сообщения пользователя может быть описан как последовательность преобразований. Пусть $m$ -- текст сообщения. На первом этапе NLU-компонент выполняет отображение $\phi(m) = (i, E)$, где $i \in \mathcal{I}$ -- распознанный интент (например, «поиск_документа», «статус_дела»), а $E = \{(e_j, t_j)\}$ -- множество извлечённых сущностей с их типами. На втором этапе диалоговый менеджер определяет действие $a = \pi(s, i, E)$, где $s$ -- текущее состояние диалога и $\pi$ -- политика управления. На третьем этапе действие выполняется (запрос к API) и формируется ответ.

### 6.2. Реализация на aiogram 3.x

Фреймворк aiogram 3.x реализует полностью асинхронное взаимодействие с Telegram Bot API, что обеспечивает эффективную обработку множества одновременных диалогов. Система роутеров (Router) позволяет организовать обработчики сообщений в модульную структуру, а механизм конечного автомата (FSM -- Finite State Machine) обеспечивает управление многоэтапными диалогами, в которых бот последовательно уточняет параметры запроса.

### 6.3. Обработка естественного языка в чат-боте

Для повышения качества интерпретации пользовательских запросов чат-бот интегрируется с NLP-компонентами. Распознавание намерений реализуется посредством классификатора текстов, обученного на размеченном корпусе типичных запросов. Извлечение сущностей (номер дела, дата, наименование суда) выполняется NER-моделью. Для запросов на естественном языке применяется семантический поиск, передающий текст запроса в эндпойнт гибридного поиска backend API.

---


In [ ]:
# ============================================================
# Демонстрация: структура Telegram-бота юридической ИПС
# Фреймворк: aiogram 3.x (структурный пример)
# ============================================================

# Примечание: для запуска необходим pip install aiogram aiohttp

bot_structure = '''
# === Состояния FSM (конечный автомат диалога) ===
from aiogram.fsm.state import State, StatesGroup

class SearchStates(StatesGroup):
    waiting_query    = State()    # ожидание текста запроса
    waiting_filter   = State()    # ожидание выбора фильтра
    showing_results  = State()    # просмотр результатов

# === Обработчик команды /start ===
@router.message(CommandStart())
async def cmd_start(message):
    await message.answer(
        "Я -- бот юридической ИПС.\n"
        "Напишите запрос, например:\n"
        "  'банкротство субсидиарная ответственность'\n"
        "  'дело А40-12345/2024'"
    )

# === Обработчик текстовых сообщений ===
@router.message(F.text)
async def handle_search(message, state):
    query = message.text.strip()
    await message.answer_chat_action("typing")

    # Запрос к backend API через aiohttp
    async with aiohttp.ClientSession() as session:
        payload = {"query": query, "mode": "hybrid", "limit": 5}
        async with session.post(API_URL + "/search", json=payload) as resp:
            data = await resp.json()

    results = data.get("results", [])
    if not results:
        await message.answer("Ничего не найдено. Переформулируйте запрос.")
        return

    # Формирование ответа с инлайн-кнопками
    text = f"Найдено: {data['total']} документов\n\n"
    for i, doc in enumerate(results, 1):
        text += f"{i}. {doc['title']}\n"
        text += f"   {doc['doc_type']} | {doc['date']}\n\n"

    builder = InlineKeyboardBuilder()
    for doc in results:
        builder.button(text=f"Doc #{doc['id']}", callback_data=f"doc_{doc['id']}")
    builder.adjust(2)

    await message.answer(text, reply_markup=builder.as_markup())

# === Обработчик кнопки "Подробнее" ===
@router.callback_query(F.data.startswith("doc_"))
async def show_document(callback):
    doc_id = int(callback.data.split("_")[1])
    # Запрос полного документа через API
    # ... (аналогично поиску)
    await callback.answer()
'''

print("Структура Telegram-бота юридической ИПС (aiogram 3.x)")
print("=" * 60)
print(bot_structure)
print("\nКомпоненты бота:")
print("  1. FSM (конечный автомат) для управления диалогом")
print("  2. aiohttp для асинхронных запросов к backend API")
print("  3. InlineKeyboardBuilder для навигации по результатам")
print("  4. Поддержка команд /start, /help и текстовых запросов")

## 7. Обеспечение безопасности и производительности

Юридическая информационная система обрабатывает данные повышенной чувствительности: персональные данные участников судебных процессов, содержание судебных актов по незавершённым делам, сведения о коммерческих спорах и финансовых обязательствах. В таких условиях обеспечение безопасности является не факультативным дополнением, а обязательным требованием, определяющим саму возможность промышленной эксплуатации системы.

### 7.1. Аутентификация и авторизация

Аутентификация (установление личности пользователя) и авторизация (определение прав доступа) реализуются посредством стандартного протокола JWT (JSON Web Token). При успешном входе в систему сервер генерирует подписанный токен, содержащий идентификатор пользователя, его роль и срок действия. Токен передаётся клиентом в заголовке $\texttt{Authorization: Bearer <token>}$ каждого последующего запроса. Сервер верифицирует подпись токена и извлекает из него информацию о пользователе без обращения к базе данных, что обеспечивает высокую производительность.

Формально JWT-токен представляет собой строку вида $\texttt{header.payload.signature}$, где $\texttt{header}$ содержит алгоритм подписи, $\texttt{payload}$ -- полезную нагрузку (claims), а $\texttt{signature} = \text{HMAC}(\texttt{header} \| \texttt{payload}, \texttt{secret})$ обеспечивает целостность и аутентичность токена.

### 7.2. Защита от атак

Веб-приложение должно быть защищено от стандартного набора атак, описанного в рейтинге OWASP Top 10. SQL-инъекции предотвращаются использованием параметризованных запросов (которые автоматически обеспечивает ORM SQLAlchemy). Межсайтовый скриптинг (XSS) предотвращается экранированием пользовательского ввода при отображении в HTML. Подделка межсайтовых запросов (CSRF) предотвращается использованием токенов CSRF для форм и проверкой заголовка $\texttt{Origin}$. Атаки типа «перебор» (brute force) предотвращаются ограничением частоты запросов (rate limiting) и механизмом временной блокировки учётных записей.

### 7.3. Защита персональных данных

Обработка персональных данных участников судебных процессов регулируется Федеральным законом No 152-ФЗ «О персональных данных». Система должна обеспечивать обезличивание (деперсонализацию) данных в публичных интерфейсах, разграничение доступа к чувствительным полям на уровне API-эндпойнтов, ведение журнала доступа к персональным данным с фиксацией субъекта, объекта и времени операции. При передаче данных между компонентами системы применяется шифрование TLS.

### 7.4. Производительность и кэширование

Для обеспечения приемлемого времени отклика при высокой нагрузке применяется многоуровневое кэширование. На уровне приложения Redis используется для кэширования результатов частых поисковых запросов с настраиваемым временем жизни (TTL). На уровне HTTP применяются заголовки кэширования ($\texttt{Cache-Control}$, $\texttt{ETag}$) для статических ресурсов. На уровне СУБД материализованные представления обеспечивают предварительное вычисление аналитических агрегатов.

Для оценки производительности используются перцентильные метрики: $p_{50}$, $p_{95}$, $p_{99}$ -- характеризующие типичную и наихудшую задержку. Целевые значения для юридической системы: $p_{50} < 200$ мс, $p_{95} < 1$ с, $p_{99} < 3$ с.

---


In [ ]:
# ============================================================
# Демонстрация: структура JWT-токена
# ============================================================
import json, base64

# Составные части JWT
header_data = {'alg': 'HS256', 'typ': 'JWT'}
payload_data = {'sub': 'user_42', 'role': 'analyst', 'exp': 1735689600}

header_b64 = base64.urlsafe_b64encode(json.dumps(header_data).encode()).decode().rstrip('=')
payload_b64 = base64.urlsafe_b64encode(json.dumps(payload_data).encode()).decode().rstrip('=')

print('Структура JWT-токена')
print('=' * 60)
print(f'  Header (base64):  {header_b64}')
print(f'  Payload (base64): {payload_b64}')
print(f'  Signature:        <HMAC-SHA256(header.payload, secret)>')
print(f'  Полный токен:     {header_b64}.{payload_b64}.<signature>')
print()
print('Декодированный payload:')
print(f'  {payload_data}')
print()
print('В FastAPI защита эндпойнта реализуется через Depends():')
print('  @app.get("/api/v1/admin/stats")')
print('  async def admin_stats(user=Depends(get_current_user)):')
print('      if user["role"] != "admin":')
print('          raise HTTPException(403, "Insufficient permissions")')


## 8. Тестирование и развёртывание

Обеспечение качества программного продукта требует систематического тестирования на нескольких уровнях. В контексте юридической информационной системы тестирование приобретает особую важность, поскольку ошибки в результатах поиска или отображении документов могут привести к некорректным юридическим решениям.

### 8.1. Уровни тестирования

Модульные (юнит) тесты проверяют корректность работы отдельных функций и классов в изоляции от внешних зависимостей. Для серверной части юридической системы юнит-тесты охватывают валидацию Pydantic-схем, корректность преобразования параметров поиска, логику ранжирования результатов (в частности, алгоритм RRF) и обработку граничных случаев.

Интеграционные тесты проверяют взаимодействие между компонентами: корректность SQL-запросов к реальной базе данных, работоспособность полнотекстового поиска, взаимодействие между API-эндпойнтами и слоем доступа к данным. FastAPI предоставляет встроенный $\texttt{TestClient}$ для имитации HTTP-запросов без запуска реального сервера, что существенно ускоряет выполнение тестов.

Сквозные (end-to-end) тесты проверяют систему в целом с точки зрения конечного пользователя: ввод поискового запроса в интерфейсе, получение результатов, переход к детальному просмотру документа. Для автоматизации сквозных тестов применяются инструменты Playwright или Selenium, имитирующие действия пользователя в браузере.

### 8.2. Контейнеризация и оркестрация

Контейнеризация с помощью Docker обеспечивает воспроизводимость среды выполнения и упрощает развёртывание. Каждый компонент системы (backend, frontend, база данных, Redis, NLP-сервисы) упаковывается в отдельный Docker-образ с явным описанием зависимостей. Docker Compose обеспечивает координированный запуск всех компонентов и настройку сетевого взаимодействия между ними.

Для промышленных развёртываний с высокими требованиями к доступности и масштабируемости применяется оркестратор Kubernetes, обеспечивающий автоматическое масштабирование (Horizontal Pod Autoscaler), самовосстановление при сбоях (health checks и liveness probes), а также поэтапное обновление (rolling updates) без прерывания обслуживания.

### 8.3. CI/CD-конвейер

Автоматизация процессов сборки, тестирования и развёртывания реализуется посредством CI/CD-конвейера (Continuous Integration / Continuous Deployment). Типичный конвейер включает этапы: линтинг кода, выполнение юнит- и интеграционных тестов, сборка Docker-образов, публикация образов в реестр, развёртывание на тестовом окружении, выполнение сквозных тестов и, при успешном прохождении всех проверок, развёртывание на рабочем окружении.

---


In [ ]:
# ============================================================
# Демонстрация: Docker Compose для юридической ИПС
# ============================================================

docker_compose_yaml = '''
version: "3.9"
services:
  db:
    image: pgvector/pgvector:pg16
    environment:
      POSTGRES_DB: legal_db
      POSTGRES_USER: legal_user
      POSTGRES_PASSWORD: ${DB_PASSWORD}
    volumes:
      - pgdata:/var/lib/postgresql/data
    ports: ["5432:5432"]

  redis:
    image: redis:7-alpine
    ports: ["6379:6379"]

  backend:
    build: ./backend
    environment:
      DATABASE_URL: postgresql+asyncpg://legal_user:${DB_PASSWORD}@db/legal_db
      REDIS_URL: redis://redis:6379
      JWT_SECRET: ${JWT_SECRET}
    ports: ["8000:8000"]
    depends_on: [db, redis]

  frontend:
    build: ./frontend
    ports: ["3000:80"]
    depends_on: [backend]

volumes:
  pgdata:
'''

print('Docker Compose для юридической ИПС')
print('=' * 60)
print(docker_compose_yaml)

print('Команды запуска:')
print('  docker compose up -d          # запуск всех сервисов')
print('  docker compose logs backend   # просмотр логов')
print('  pytest tests/ -v              # запуск тестов')
print('  docker compose down           # остановка')


## 9. Улучшение пользовательского опыта

Пользовательский опыт (User Experience, UX) определяет общее впечатление пользователя от взаимодействия с системой и непосредственно влияет на частоту и эффективность использования инструмента. В юридическом контексте качество UX приобретает особое значение, поскольку пользователями системы являются как профессиональные юристы с высокими требованиями к точности и полноте поиска, так и граждане без специальной подготовки, для которых интуитивность интерфейса является определяющим фактором.

### 9.1. Адаптивный дизайн и доступность

Адаптивный дизайн (responsive design) обеспечивает корректное отображение интерфейса на устройствах с различными размерами экрана: настольных компьютерах, планшетах и смартфонах. Реализация адаптивности основана на медиа-запросах CSS и гибких сетках (CSS Grid, Flexbox), которые автоматически перестраивают расположение элементов. Для юридической системы особенно важна корректная работа на мобильных устройствах, поскольку юристы нередко обращаются к системе из зала суда.

Доступность (accessibility) подразумевает обеспечение возможности использования системы людьми с ограниченными возможностями, что регулируется стандартом WCAG 2.1. Основные требования включают поддержку программ чтения с экрана (screen readers), навигацию с клавиатуры, достаточный цветовой контраст и наличие текстовых альтернатив для графических элементов.

### 9.2. Интеллектуальные функции интерфейса

Современные юридические поисковые системы реализуют ряд интеллектуальных функций, повышающих эффективность взаимодействия. Автодополнение (autocomplete) предлагает варианты завершения запроса по мере ввода, основываясь на частоте предыдущих запросов и терминологии предметной области. Фасетная навигация позволяет пользователю итеративно уточнять результаты поиска, добавляя фильтры по категории, суду, временному периоду и типу документа. Подсветка найденных терминов в тексте документа облегчает визуальную оценку релевантности. Механизм уведомлений информирует пользователя о новых документах по сохранённым поисковым запросам.

Важным элементом UX является обработка пустых результатов поиска. Вместо лаконичного сообщения «Ничего не найдено» система должна предлагать альтернативные формулировки запроса, расширение параметров поиска или ссылки на смежные тематические категории.

---


## 10. Заключение

В настоящей лекции рассмотрен полный цикл разработки пользовательских интерфейсов для юридической информационно-поисковой системы: от архитектурного проектирования до практической реализации веб-приложения и диалогового агента.

Архитектура юридической информационной системы следует трёхуровневой модели, в которой слой представления (веб-приложение, чат-бот) взаимодействует со слоем бизнес-логики (FastAPI backend) через стандартизованный REST API, а слой бизнес-логики обращается к слою данных (PostgreSQL, Redis, pgvector). Такое разделение обеспечивает независимость компонентов, их автономное масштабирование и возможность замены без влияния на остальные части системы.

Технологический стек Python (FastAPI + SQLAlchemy + Pydantic) на стороне сервера обеспечивает бесшовную интеграцию с NLP-компонентами, рассмотренными в предыдущих лекциях, а также строгую валидацию данных и автоматическую генерацию документации API. Клиентская часть, реализованная как SPA на React, обеспечивает отзывчивый интерфейс с поддержкой фасетной навигации, автодополнения и подсветки результатов.

Диалоговый интерфейс (Telegram-бот на aiogram) представляет собой альтернативный канал доступа, формализованный через модель $\phi(m) = (i, E)$ распознавания намерений и извлечения сущностей. Асинхронная архитектура бота обеспечивает обслуживание множества одновременных диалогов, а механизм FSM позволяет реализовать многоэтапные сценарии уточнения запроса.

Обеспечение безопасности реализуется на нескольких уровнях: JWT-аутентификация, защита от OWASP Top 10, соблюдение требований ФЗ-152 о персональных данных. Производительность системы обеспечивается многоуровневым кэшированием и оценивается через перцентильные метрики $p_{50}$, $p_{95}$, $p_{99}$.

---


## Контрольные вопросы для самопроверки

1. Опишите трёхуровневую архитектуру юридической информационно-поисковой системы. Какие функции выполняет каждый уровень и какие протоколы используются для взаимодействия между ними? Почему разделение на слои обеспечивает масштабируемость и сопровождаемость системы?

2. Объясните преимущества гибридного поиска по сравнению с чисто полнотекстовым или чисто семантическим. Опишите алгоритм Reciprocal Rank Fusion (RRF) и объясните роль параметра $k$ в формуле $\text{RRF}(d) = \sum_{r} \frac{1}{k + \text{rank}_r(d)}$.

3. Охарактеризуйте архитектуру чат-бота с точки зрения NLU-компонента. Что представляют собой интенты и сущности в контексте юридического диалогового агента? Приведите примеры интентов и соответствующих им сущностей для системы поиска судебных документов.

4. Какие механизмы обеспечения безопасности необходимо реализовать в юридической веб-системе? Опишите принцип работы JWT-аутентификации и объясните, почему токенный подход предпочтительнее сессионного для масштабируемых систем.

5. Опишите структуру CI/CD-конвейера для юридической информационной системы. Какие этапы включает конвейер и какие инструменты используются на каждом этапе? Почему контейнеризация (Docker) является важным элементом стратегии развёртывания?

---

### Задания для самостоятельной работы

1. Реализуйте REST API на FastAPI с эндпойнтами для поиска документов (полнотекстовый и фасетный), получения документа по идентификатору и добавления нового документа. Используйте SQLAlchemy ORM для доступа к базе данных, созданной в рамках Лекции 6. Обеспечьте автоматическую генерацию документации Swagger.

2. Разработайте Telegram-бота на aiogram 3.x, который принимает текстовые запросы, передаёт их в REST API из задания 1 и отображает результаты поиска с инлайн-кнопками. Реализуйте FSM-диалог для уточнения параметров поиска (тип документа, временной диапазон).

3. Напишите набор тестов (не менее 5 юнит-тестов и 3 интеграционных) для REST API из задания 1, используя pytest и httpx. Тесты должны проверять корректность поиска, валидацию входных данных, обработку ошибок (404, 422) и корректность пагинации.

---

### Рекомендуемые источники

FastAPI Documentation. URL: https://fastapi.tiangolo.com

aiogram Documentation. URL: https://docs.aiogram.dev

React Documentation. URL: https://react.dev

Fielding R.T. *Architectural Styles and the Design of Network-based Software Architectures* (PhD Dissertation, University of California, Irvine, 2000).

OWASP Top 10. URL: https://owasp.org/www-project-top-ten/

Docker Documentation. URL: https://docs.docker.com

Pydantic Documentation. URL: https://docs.pydantic.dev

SQLAlchemy Documentation. URL: https://docs.sqlalchemy.org
